## Environment Setup and Data Preparation

This cell initializes the workspace by importing all necessary dependencies and transferring the dataset from Google Drive to the local Colab storage to optimize read speeds.

**Main operations performed in this block:**

1. **Library Imports:** Loads the core modules for the entire project, categorized by domain:
   * **Audio Processing:** `librosa`, `soundfile`, and `torchaudio` for loading and manipulating signals.
   * **Machine & Deep Learning:** `scikit-learn` for traditional models, alongside `TensorFlow`, `PyTorch`, and `Transformers` (HuggingFace) for advanced models (Wav2Vec2, AST).
   * **Data Science & Visualization:** `pandas`, `numpy`, `matplotlib`, `seaborn`, and `shap` for data handling and model interpretability.
2. **Google Drive Access:** Mounts the personal Drive to read the archived files.
3. **Dataset Extraction:** Creates a local working directory (`/content/dataset/`) and extracts the audio files (`wav.zip`) and the repository files containing the scores (`Coco-Nut-main.zip`) into it.

In [ ]:
# Standard Library Imports
import os
import tempfile
import urllib.request
import warnings
import zipfile
from pathlib import Path

# Cloud / Environment Imports
from google.colab import drive

# Data Science, Math & Utility
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from tqdm.auto import tqdm

# Audio Processing
import librosa
import soundfile as sf

# Visualization & Interpretability
import matplotlib.pyplot as plt
import seaborn as sns
import shap

# Machine Learning & Scikit-Learn
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.manifold import TSNE
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV, GroupKFold, KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

# Deep Learning (TensorFlow)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Deep Learning (PyTorch & HuggingFace)
import torch
import torchaudio
from transformers import ASTFeatureExtractor, ASTModel, Wav2Vec2Model, Wav2Vec2Processor

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Create a local working directory on Colab
os.makedirs('/content/dataset', exist_ok=True)

# Unzip the audio files from the 'Coco-nut' folder
print("Extracting audio files...")
wav_zip_path = '/content/drive/MyDrive/Coco-nut/wav.zip'
with zipfile.ZipFile(wav_zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset/')

# Unzip the GitHub repository (scores) from the 'Coco-nut' folder
print("Extracting GitHub repository (scores)...")
repo_zip_path = '/content/drive/MyDrive/Coco-nut/Coco-Nut-main.zip'
with zipfile.ZipFile(repo_zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset/')

print("Extraction complete")

## Data Preprocessing and Alignment

This cell configures the project directories, processes the raw listener ratings, and maps the target scores to the physical audio files. It also subsamples the dataset to ensure manageable computation times.

**Main operations performed in this block:**

1. **Directory Configuration:** Establishes fixed paths for caching, results, and dataset locations, creating them if they do not exist.
2. **Score Aggregation:** Loads the `train`, `validation`, and `test` CSV files. It isolates the listener rating columns, calculates the mean score for each audio track, handles missing values, and standardizes the column names for the machine learning pipeline.
3. **File Mapping:** Scans the extracted directory for `.wav` files, stripping leading zeros from filenames to ensure perfect matching with the IDs found in the CSV files. Any records lacking a physical audio file are dropped.
4. **Dataset Subsampling:** To optimize processing time, the dataset is capped at exactly 1,000 samples with a specific distribution (700 for training, 150 for validation, and 150 for testing). Finally, summary statistics for the processed splits are displayed.

In [ ]:
# Project constants and directory configuration
SEED = 42
SAMPLE_RATE = 16000
PROJECT_DIR = Path("./voice_quality_project")
RESULTS_DIR = PROJECT_DIR / "results"
CACHE_DIR = PROJECT_DIR / "cache"

for directory in [CACHE_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Coco-Nut specific paths
DATASET_ROOT = Path("/content/dataset")
WAV_DIR = DATASET_ROOT / "wav"
HUMORESQUE_DIR = DATASET_ROOT / "Coco-Nut-main" / "humoresque"

# Function to load humoresque CSVs, group by audio ID, and calculate the mean score safely
def load_and_aggregate_split(file_path, split_name):
    df = pd.read_csv(file_path)

    # Extract only the columns that actually contain the ratings (rate0, rate1, rate2...)
    rate_cols = [c for c in df.columns if c.startswith('rate')]

    # Convert only the rate columns to numeric
    for col in rate_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Calculate the mean across the valid listener rates for each row
    df['row_mean'] = df[rate_cols].mean(axis=1, skipna=True)

    # Group by audio ID ('id' column) to handle duplicate subsets per audio
    aggregated = df.groupby('id')['row_mean'].mean().reset_index()

    # Rename columns to match the ML pipeline expectations
    aggregated.rename(columns={'id': "utterance_id", 'row_mean': "target_score"}, inplace=True)

    # Clean up IDs (ensure they are strings without leading/trailing spaces)
    aggregated["utterance_id"] = aggregated["utterance_id"].astype(str).str.strip()
    aggregated["split"] = split_name

    # Drop any rows where the score is missing
    return aggregated.dropna(subset=['target_score'])

print("Loading and aggregating humoresque scores (calculating mean attractiveness)...")
train_df = load_and_aggregate_split(HUMORESQUE_DIR / "train.csv", "train")
val_df = load_and_aggregate_split(HUMORESQUE_DIR / "valid.csv", "validation")
test_df = load_and_aggregate_split(HUMORESQUE_DIR / "test.csv", "test")

dataset_all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

# Map physical audio files on disk
print("Mapping audio files to their labels...")
audio_files = [p for p in WAV_DIR.rglob("*.wav") if not p.name.startswith("._")]

# We strip leading zeros from the filename stem to perfectly match the CSV IDs
audio_mapping = {str(int(p.stem)): str(p) for p in audio_files}

# Format the dataframe IDs exactly the same way to guarantee a perfect match
dataset_all_df["utterance_id"] = dataset_all_df["utterance_id"].apply(lambda x: str(int(x)))

# Assign the audio path
dataset_all_df["audio_path"] = dataset_all_df["utterance_id"].map(audio_mapping)
dataset_all_df["audio_found"] = dataset_all_df["audio_path"].notna()

# Filter only records with physically present audio
dataset_df = dataset_all_df[dataset_all_df["audio_found"]].copy().reset_index(drop=True)

# Capping the dataset to 1000 samples
if len(dataset_df) > 1000:
    print(f"\nOriginal labeled audio found: {len(dataset_df)}. Capping to 1000 samples...")

    # Target distribution for our 1000 subset
    cap_sizes = {
        'train': 700,
        'validation': 150,
        'test': 150
    }

    sampled_dfs = []
    for split_name, df_split in dataset_df.groupby("split"):
        n_samples = min(cap_sizes.get(split_name, len(df_split)), len(df_split))
        sampled_dfs.append(df_split.sample(n=n_samples, random_state=SEED))

    dataset_df = pd.concat(sampled_dfs).reset_index(drop=True)

print(f"\nLOADED DATASET STATISTICS")
print(f"Total audio tracks ready for the pipeline: {len(dataset_df)}")
print(dataset_df.groupby("split")["target_score"].agg(["count", "mean", "std", "min", "max"]).round(3))

## Acoustic Feature Extraction

This cell extracts a comprehensive set of traditional handcrafted acoustic features from the audio files using `librosa`. Since audio signals are time-varying, it converts frame-level data into fixed-size feature vectors that traditional machine learning algorithms can process.

**Main operations performed in this block:**

1. **Statistical Summarization:** Defines a helper function (`summarize`) to compute global statistics (mean, standard deviation, median, 10th percentile, and 90th percentile) across the duration of each audio track.
2. **Signal Processing Pipeline:** Loads each audio file (normalized to 16kHz mono) and calculates a rich set of audio descriptors, including:
   * **Timbral features:** MFCCs (and their first/second derivatives), spectral centroid, bandwidth, rolloff, flatness, contrast, and entropy.
   * **Prosodic & Energy features:** Fundamental frequency (F0/pitch), Root Mean Square (RMS) energy, Zero-Crossing Rate (ZCR), and harmonic ratio.
3. **Caching System:** Implements a caching mechanism (`handcrafted_features.pkl`) to save extracted features to the local disk. This drastically reduces execution time on subsequent notebook runs and safely logs any corrupted files.
4. **Data Integration & Cleaning:** Merges the newly extracted features with the main dataset and automatically removes any constant features (zero variance based on the training split) to prevent numerical instability during model training.

In [ ]:
warnings.filterwarnings("ignore")

# Helper functions for statistics and preprocessing
def summarize(values, prefix):
    """Calculates basic statistics for an array of values."""
    values = np.asarray(values, dtype=np.float32)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return {f"{prefix}_{stat}": 0.0 for stat in ["mean", "std", "median", "p10", "p90"]}

    return {
        f"{prefix}_mean": float(np.mean(values)),
        f"{prefix}_std": float(np.std(values)),
        f"{prefix}_median": float(np.median(values)),
        f"{prefix}_p10": float(np.percentile(values, 10)),
        f"{prefix}_p90": float(np.percentile(values, 90))
    }

def spectral_entropy(magnitude):
    """Calculates normalized spectral entropy."""
    probabilities = magnitude / (np.sum(magnitude, axis=0, keepdims=True) + 1e-12)
    entropy = -np.sum(probabilities * np.log2(probabilities + 1e-12), axis=0)
    return entropy / np.log2(magnitude.shape[0])

# Acoustic feature extraction pipeline
def extract_handcrafted_features(path, sample_rate=SAMPLE_RATE):
    """Loads the audio and extracts the complete set of classic features."""
    # True 16kHz mono loader with normalization
    waveform, _ = librosa.load(path, sr=sample_rate, mono=True)
    waveform = waveform.astype(np.float32)
    waveform = waveform - np.mean(waveform)
    waveform = np.clip(waveform, -1.0, 1.0)

    # Feature dictionary initialization
    features = {"duration_seconds_feature": len(waveform) / sample_rate}

    # Descriptor Extraction
    rms = librosa.feature.rms(y=waveform, frame_length=1024, hop_length=256)[0]
    zcr = librosa.feature.zero_crossing_rate(y=waveform, frame_length=1024, hop_length=256)[0]

    magnitude = np.abs(librosa.stft(waveform, n_fft=1024, hop_length=256))
    centroid = librosa.feature.spectral_centroid(S=magnitude, sr=sample_rate)[0]
    bandwidth = librosa.feature.spectral_bandwidth(S=magnitude, sr=sample_rate)[0]
    rolloff = librosa.feature.spectral_rolloff(S=magnitude, sr=sample_rate, roll_percent=0.85)[0]
    flatness = librosa.feature.spectral_flatness(S=magnitude)[0]
    contrast = librosa.feature.spectral_contrast(S=magnitude, sr=sample_rate)
    entropy = spectral_entropy(magnitude)

    mfcc = librosa.feature.mfcc(y=waveform, sr=sample_rate, n_mfcc=13, n_fft=1024, hop_length=256)
    mfcc_delta = librosa.feature.delta(mfcc)
    mfcc_delta2 = librosa.feature.delta(mfcc, order=2)

    f0 = librosa.yin(waveform, fmin=50, fmax=500, sr=sample_rate, frame_length=1024, hop_length=256)
    f0 = f0[(f0 >= 50) & (f0 <= 500)]

    harmonic = librosa.effects.harmonic(waveform)
    harmonic_ratio = np.sqrt(np.mean(harmonic ** 2)) / (np.sqrt(np.mean(waveform ** 2)) + 1e-12)
    features["harmonic_ratio"] = float(harmonic_ratio)

    # Statistics aggregation
    for name, vals in zip(["rms", "zcr", "spectral_centroid", "spectral_bandwidth",
                           "spectral_rolloff", "spectral_flatness", "spectral_entropy", "f0"],
                          [rms, zcr, centroid, bandwidth, rolloff, flatness, entropy, f0]):
        features.update(summarize(vals, name))

    for index in range(mfcc.shape[0]):
        features.update(summarize(mfcc[index], f"mfcc_{index + 1}"))
        features.update(summarize(mfcc_delta[index], f"mfcc_delta_{index + 1}"))
        features.update(summarize(mfcc_delta2[index], f"mfcc_delta2_{index + 1}"))

    for index in range(contrast.shape[0]):
        features.update(summarize(contrast[index], f"spectral_contrast_{index + 1}"))

    return features

# Execution with Caching
HANDCRAFTED_CACHE = CACHE_DIR / "handcrafted_features.pkl"

if HANDCRAFTED_CACHE.exists():
    print(f"Loading features from cache: {HANDCRAFTED_CACHE}")
    handcrafted_df = pd.read_pickle(HANDCRAFTED_CACHE)
else:
    feature_records = []
    failed_records = []

    for row in tqdm(dataset_df[["utterance_id", "audio_path"]].itertuples(index=False),
                    total=len(dataset_df), desc="Extracting classic features"):
        try:
            record = {"utterance_id": row.utterance_id}
            record.update(extract_handcrafted_features(row.audio_path))
            feature_records.append(record)
        except Exception as error:
            failed_records.append({"utterance_id": row.utterance_id, "error": str(error)})

    handcrafted_df = pd.DataFrame(feature_records)
    handcrafted_df.to_pickle(HANDCRAFTED_CACHE)

    if failed_records:
        pd.DataFrame(failed_records).to_csv(RESULTS_DIR / "handcrafted_failures.csv", index=False)
        print(f"Warning: {len(failed_records)} files generated errors. Details in results/handcrafted_failures.csv")

# Final merge with main dataset and removal of constant features
model_df = dataset_df.merge(handcrafted_df, on="utterance_id", how="inner")
feature_columns = [col for col in handcrafted_df.columns if col != "utterance_id"]

train_feature_variance = model_df.loc[model_df["split"] == "train", feature_columns].var()
feature_columns = train_feature_variance[train_feature_variance > 1e-12].index.tolist()

print(f"\nExtraction completed")
print(f"Final dataset dimensions: {model_df.shape}")
print(f"Number of valid classic features extracted: {len(feature_columns)}")

## Deep Feature Extraction and Exploratory Data Analysis

This cell leverages state-of-the-art Self-Supervised Learning (SSL) models to extract deep acoustic embeddings and visualizes their latent spaces. It utilizes Hugging Face's `transformers` library to process the audio through both Wav2Vec 2.0 and Audio Spectrogram Transformer (AST) architectures.

**Main operations performed in this block:**

1. **Model Initialization:** Loads the pre-trained Wav2Vec 2.0 and AST models from local storage and optimally maps them to the GPU (if available) for accelerated inference.
2. **Deep Embedding Extraction:** Processes the audio signals (resampling to 16kHz and converting to mono) and passes them through the networks. It extracts global feature representations (768-dimensional vectors) using mean pooling for Wav2Vec 2.0 and the native pooler output for AST.
3. **Caching and Alignment:** Saves the extracted high-dimensional matrices to the local disk as `.npy` files to prevent redundant computations on subsequent runs, and accurately aligns the feature arrays with the main dataset structure.
4. **Exploratory Visualizations (EDA):** Conducts dimensionality reduction (PCA and t-SNE) and unsupervised clustering (K-Means) on the standardized embeddings for both models. It generates comparative scatter plots to investigate how the latent spaces naturally map and correlate with the target attractiveness scores.

In [ ]:
warnings.filterwarnings("ignore")

# DEEP FEATURE EXTRACTION (SSL)
print("\nDEEP FEATURE EXTRACTION")

# Setup device (GPU recommended for Hugging Face models)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load Pre-trained Models from local Colab storage
print("Loading Wav2Vec 2.0 and AST models from local files...")
w2v_path = "/content/wav2vec2"
ast_path = "/content/AST"

w2v_processor = Wav2Vec2Processor.from_pretrained(w2v_path, local_files_only=True)
w2v_model = Wav2Vec2Model.from_pretrained(w2v_path, local_files_only=True).to(device)

ast_processor = ASTFeatureExtractor.from_pretrained(ast_path, local_files_only=True)
ast_model = ASTModel.from_pretrained(ast_path, local_files_only=True).to(device)

def extract_deep_embeddings(audio_path, target_sr=16000):
    """Loads audio and extracts embeddings from Wav2Vec2 and AST."""
    waveform, sr = torchaudio.load(audio_path)

    # Resample if needed
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sr)
        waveform = resampler(waveform)

    # Convert stereo to mono
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    waveform_np = waveform.squeeze().numpy()

    # Wav2Vec 2.0 Extraction
    w2v_inputs = w2v_processor(waveform_np, sampling_rate=target_sr, return_tensors="pt").to(device)
    with torch.inference_mode():
        w2v_outputs = w2v_model(**w2v_inputs)
    w2v_embedding = w2v_outputs.last_hidden_state.mean(dim=1).cpu().numpy().flatten()

    # AST Extraction
    ast_inputs = ast_processor(waveform_np, sampling_rate=target_sr, return_tensors="pt").to(device)
    with torch.inference_mode():
        ast_outputs = ast_model(**ast_inputs)
    ast_embedding = ast_outputs.pooler_output.cpu().numpy().flatten()

    return w2v_embedding, ast_embedding

# We use the exactly 1000 samples we already prepared in the first cell
deep_df = model_df.copy()
print(f"Extracting on our perfectly stratified {len(deep_df)} samples...")

# Extraction with caching
DEEP_W2V_CACHE = CACHE_DIR / "wav2vec2_embeddings.npy"
DEEP_AST_CACHE = CACHE_DIR / "ast_embeddings.npy"
DEEP_IDS_CACHE = CACHE_DIR / "deep_utterance_ids.npy"

if DEEP_W2V_CACHE.exists() and DEEP_AST_CACHE.exists() and DEEP_IDS_CACHE.exists():
    print("Loading Deep Embeddings from cache...")
    w2v_matrix = np.load(DEEP_W2V_CACHE)
    ast_matrix = np.load(DEEP_AST_CACHE)
    deep_utterance_ids = np.load(DEEP_IDS_CACHE, allow_pickle=True)
else:
    w2v_list, ast_list, deep_utterance_ids = [], [], []
    print("Extracting Deep Embeddings (This will take a few minutes)...")

    for row in tqdm(deep_df[["utterance_id", "audio_path"]].itertuples(index=False), total=len(deep_df)):
        try:
            w2v_emb, ast_emb = extract_deep_embeddings(row.audio_path)
        except Exception as e:
            print(f"Error processing {row.utterance_id}: {e}")
            w2v_emb, ast_emb = np.zeros(768), np.zeros(768)

        w2v_list.append(w2v_emb)
        ast_list.append(ast_emb)
        deep_utterance_ids.append(row.utterance_id)

    w2v_matrix = np.vstack(w2v_list).astype(np.float32)
    ast_matrix = np.vstack(ast_list).astype(np.float32)
    deep_utterance_ids = np.asarray(deep_utterance_ids)

    np.save(DEEP_W2V_CACHE, w2v_matrix)
    np.save(DEEP_AST_CACHE, ast_matrix)
    np.save(DEEP_IDS_CACHE, deep_utterance_ids)

# Align indices on the dataset
deep_index = {uid: i for i, uid in enumerate(deep_utterance_ids)}
valid_deep_mask = model_df["utterance_id"].isin(deep_utterance_ids)
filtered_model_df = model_df[valid_deep_mask].copy()

deep_order = filtered_model_df["utterance_id"].map(deep_index).to_numpy()
w2v_all = w2v_matrix[deep_order]
ast_all = ast_matrix[deep_order]

# Train slice reused for the exploratory analysis
train_mask_eda = (filtered_model_df["split"] == "train").to_numpy()
X_train_w2v = w2v_all[train_mask_eda]
X_train_ast = ast_all[train_mask_eda]

y_train_score = filtered_model_df.loc[filtered_model_df["split"] == "train", "target_score"].to_numpy()


# EXPLORATORY DATA ANALYSIS (Wav2Vec2)
print("\nEXPLORATORY ANALYSIS (Wav2Vec2 Embeddings)")

scaler_w2v = StandardScaler()
X_scaled_w2v = scaler_w2v.fit_transform(X_train_w2v)

print("Computing PCA and t-SNE for Wav2Vec2...")
pca_w2v = PCA(n_components=2, random_state=SEED)
pca_coords_w2v = pca_w2v.fit_transform(X_scaled_w2v)

tsne_w2v = TSNE(n_components=2, perplexity=30, random_state=SEED, n_jobs=-1)
tsne_coords_w2v = tsne_w2v.fit_transform(X_scaled_w2v)

print("Applying K-Means clustering for Wav2Vec2...")
kmeans_w2v = KMeans(n_clusters=3, random_state=SEED, n_init=10)
cluster_labels_w2v = kmeans_w2v.fit_predict(X_scaled_w2v)

fig1, axes1 = plt.subplots(1, 3, figsize=(18, 6))

scatter1 = axes1[0].scatter(pca_coords_w2v[:, 0], pca_coords_w2v[:, 1], c=y_train_score, cmap="magma", alpha=0.7, s=15)
axes1[0].set_title(f"PCA of Wav2Vec2 (PC1: {pca_w2v.explained_variance_ratio_[0]:.2%})")
axes1[0].set_xlabel("Principal Component 1")
axes1[0].set_ylabel("Principal Component 2")
fig1.colorbar(scatter1, ax=axes1[0], label="Attractiveness Score")

scatter2 = axes1[1].scatter(tsne_coords_w2v[:, 0], tsne_coords_w2v[:, 1], c=y_train_score, cmap="magma", alpha=0.7, s=15)
axes1[1].set_title("t-SNE of Wav2Vec2 Embeddings")
axes1[1].set_xlabel("t-SNE Dimension 1")
axes1[1].set_ylabel("t-SNE Dimension 2")
fig1.colorbar(scatter2, ax=axes1[1], label="Attractiveness Score")

scatter3 = axes1[2].scatter(tsne_coords_w2v[:, 0], tsne_coords_w2v[:, 1], c=cluster_labels_w2v, cmap="viridis", alpha=0.7, s=15)
axes1[2].set_title("K-Means Clustering on Embeddings (K=3)")
axes1[2].set_xlabel("t-SNE Dimension 1")
axes1[2].set_ylabel("t-SNE Dimension 2")
fig1.colorbar(scatter3, ax=axes1[2], label="Cluster ID")

plt.tight_layout()
plt.show()


# EXPLORATORY DATA ANALYSIS (AST)
print("\nEXPLORATORY ANALYSIS (AST Embeddings)")

scaler_ast = StandardScaler()
X_scaled_ast = scaler_ast.fit_transform(X_train_ast)

print("Computing PCA and t-SNE for AST...")
pca_ast = PCA(n_components=2, random_state=SEED)
pca_coords_ast = pca_ast.fit_transform(X_scaled_ast)

tsne_ast = TSNE(n_components=2, perplexity=30, random_state=SEED, n_jobs=-1)
tsne_coords_ast = tsne_ast.fit_transform(X_scaled_ast)

print("Applying K-Means clustering for AST...")
kmeans_ast = KMeans(n_clusters=3, random_state=SEED, n_init=10)
cluster_labels_ast = kmeans_ast.fit_predict(X_scaled_ast)

fig2, axes2 = plt.subplots(1, 3, figsize=(18, 6))

scatter_ast1 = axes2[0].scatter(pca_coords_ast[:, 0], pca_coords_ast[:, 1], c=y_train_score, cmap="magma", alpha=0.7, s=15)
axes2[0].set_title(f"PCA of AST (PC1: {pca_ast.explained_variance_ratio_[0]:.2%})")
axes2[0].set_xlabel("Principal Component 1")
axes2[0].set_ylabel("Principal Component 2")
fig2.colorbar(scatter_ast1, ax=axes2[0], label="Attractiveness Score")

scatter_ast2 = axes2[1].scatter(tsne_coords_ast[:, 0], tsne_coords_ast[:, 1], c=y_train_score, cmap="magma", alpha=0.7, s=15)
axes2[1].set_title("t-SNE of AST Embeddings")
axes2[1].set_xlabel("t-SNE Dimension 1")
axes2[1].set_ylabel("t-SNE Dimension 2")
fig2.colorbar(scatter_ast2, ax=axes2[1], label="Attractiveness Score")

scatter_ast3 = axes2[2].scatter(tsne_coords_ast[:, 0], tsne_coords_ast[:, 1], c=cluster_labels_ast, cmap="viridis", alpha=0.7, s=15)
axes2[2].set_title("K-Means Clustering on AST (K=3)")
axes2[2].set_xlabel("t-SNE Dimension 1")
axes2[2].set_ylabel("t-SNE Dimension 2")
fig2.colorbar(scatter_ast3, ax=axes2[2], label="Cluster ID")

plt.tight_layout()
plt.show()

## Spectrogram Extraction and CNN-BiLSTM Training

This cell shifts the focus from feature-based machine learning to end-to-end deep learning. It converts the raw audio into 2D Log-Mel Spectrograms and trains a custom hybrid neural network (CNN-BiLSTM) to predict attractiveness scores directly from these time-frequency representations.

**Main operations performed in this block:**

1. **Spectrogram Generation:** Standardizes all audio tracks to a fixed length (padding or truncating to exactly 8 seconds) and extracts 64-band Log-Mel Spectrograms using `librosa`.
2. **Data Caching and Normalization:** Caches the generated arrays locally to save time on future runs. It then applies global Z-score normalization, calculating the mean and standard deviation exclusively from the training set, and reshapes the tensors to include a channel dimension for spatial processing.
3. **Hybrid Architecture Construction:** Uses `TensorFlow/Keras` to build a model combining two powerful architectures:
   * **CNN Blocks:** Two 2D Convolutional layers with Batch Normalization and Max Pooling to extract local acoustic textures and spatial patterns from the spectrograms.
   * **BiLSTM Layer:** A Bidirectional Long Short-Term Memory layer to process the reshaped CNN feature maps and capture the sequential, temporal dynamics of the human voice.
4. **Model Training and Evaluation:** Compiles the model with the AdamW optimizer and trains it using robust callbacks (`EarlyStopping`, `ReduceLROnPlateau`, and `ModelCheckpoint`) to prevent overfitting. Finally, it plots the learning curves (MSE) and computes the Mean Absolute Error (MAE) on the validation set.

In [ ]:
# Spectrogram Extraction Parameters
MAX_SECONDS = 8
MAX_SAMPLES = SAMPLE_RATE * MAX_SECONDS
CNN_HOP_LENGTH = 512
CNN_FRAMES = 256
N_MELS = 64

MEL_CACHE = CACHE_DIR / "cnn_logmel_subset.npy"
MEL_IDS_CACHE = CACHE_DIR / "cnn_utterance_ids_subset.npy"
MODELS_DIR = RESULTS_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Clears Keras session to avoid memory overload
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

def load_audio(path, sample_rate=SAMPLE_RATE):
    waveform, _ = librosa.load(path, sr=sample_rate, mono=True)
    waveform = waveform.astype(np.float32)
    waveform = waveform - np.mean(waveform)
    return np.clip(waveform, -1.0, 1.0)

def fixed_length_waveform(waveform, max_samples=MAX_SAMPLES):
    waveform = waveform[:max_samples]
    if len(waveform) < max_samples:
        waveform = np.pad(waveform, (0, max_samples - len(waveform)))
    return waveform.astype(np.float32)

def make_cnn_logmel(path):
    waveform = fixed_length_waveform(load_audio(path))
    mel = librosa.feature.melspectrogram(
        y=waveform, sr=SAMPLE_RATE, n_fft=1024,
        hop_length=CNN_HOP_LENGTH, n_mels=N_MELS,
        fmin=20, fmax=SAMPLE_RATE // 2, power=2.0
    )
    logmel = librosa.power_to_db(mel, ref=1.0)
    logmel = np.clip(logmel, -80.0, 20.0)
    return librosa.util.fix_length(logmel, size=CNN_FRAMES, axis=1).astype(np.float32)

# Log-Mel Extraction and Caching
if MEL_CACHE.exists() and MEL_IDS_CACHE.exists():
    print("Loading Spectrograms from cache...")
    mel_matrix = np.load(MEL_CACHE)
    mel_utterance_ids = np.load(MEL_IDS_CACHE, allow_pickle=True)
else:
    mel_features, mel_utterance_ids = [], []
    for row in tqdm(filtered_model_df[["utterance_id", "audio_path"]].itertuples(index=False),
                    total=len(filtered_model_df), desc="Extracting Mel-spectrograms"):
        mel_features.append(make_cnn_logmel(row.audio_path))
        mel_utterance_ids.append(row.utterance_id)

    mel_matrix = np.stack(mel_features).astype(np.float32)
    mel_utterance_ids = np.asarray(mel_utterance_ids)
    np.save(MEL_CACHE, mel_matrix)
    np.save(MEL_IDS_CACHE, mel_utterance_ids)

# Aligning indices with splits
mel_index_df = pd.DataFrame({"utterance_id": mel_utterance_ids, "mel_index": np.arange(len(mel_utterance_ids))})
cnn_df = filtered_model_df.merge(mel_index_df, on="utterance_id", how="inner").sort_values("mel_index").reset_index(drop=True)

# Creating tensors for training
X_mel_train_raw = mel_matrix[cnn_df.loc[cnn_df["split"] == "train", "mel_index"].to_numpy()]
X_mel_val_raw = mel_matrix[cnn_df.loc[cnn_df["split"] == "validation", "mel_index"].to_numpy()]
X_mel_test_raw = mel_matrix[cnn_df.loc[cnn_df["split"] == "test", "mel_index"].to_numpy()]

y_mel_train = cnn_df.loc[cnn_df["split"] == "train", "target_score"].to_numpy()
y_mel_val = cnn_df.loc[cnn_df["split"] == "validation", "target_score"].to_numpy()

# Global normalization on training data
mel_train_mean = X_mel_train_raw.mean()
mel_train_std = X_mel_train_raw.std() + 1e-8

X_mel_train = ((X_mel_train_raw - mel_train_mean) / mel_train_std)[..., np.newaxis]
X_mel_val = ((X_mel_val_raw - mel_train_mean) / mel_train_std)[..., np.newaxis]

print(f"\nTensor Shapes - Train: {X_mel_train.shape} | Val: {X_mel_val.shape}")

# CNN-BiLSTM Architecture
def build_cnn_bilstm(input_shape):
    inputs = keras.Input(shape=input_shape, name="logmel_input")

    x = layers.Conv2D(32, kernel_size=(3, 3), padding="same", activation="relu", name="conv_block_1")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)
    x = layers.Dropout(0.20)(x)

    x = layers.Conv2D(64, kernel_size=(3, 3), padding="same", activation="relu", name="last_conv")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Permute((2, 1, 3))(x)
    x = layers.Reshape((64, 16 * 64))(x)

    x = layers.Bidirectional(layers.LSTM(64, dropout=0.20, recurrent_dropout=0.0))(x)

    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.30)(x)

    outputs = layers.Dense(1, name="score_prediction")(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="CNN_BiLSTM")
    model.compile(optimizer=keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4), loss="mse", metrics=["mae"])
    return model

cnn_bilstm = build_cnn_bilstm(X_mel_train.shape[1:])

# Training with Callbacks
cnn_callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
    keras.callbacks.ModelCheckpoint(filepath=str(MODELS_DIR / "cnn_bilstm_best.keras"), monitor="val_loss", save_best_only=True)
]

print("\nStarting CNN-BiLSTM Training...")
history = cnn_bilstm.fit(
    X_mel_train, y_mel_train,
    validation_data=(X_mel_val, y_mel_val),
    epochs=40, batch_size=16, callbacks=cnn_callbacks, verbose=1
)

# Plot history
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Train MSE', color='#2E86AB', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation MSE', color='#A23B72', linewidth=2)
plt.title('CNN-BiLSTM Training')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.show()

# Final validation
cnn_val_pred = cnn_bilstm.predict(X_mel_val, batch_size=32, verbose=0).ravel()
cnn_mae = mean_absolute_error(y_mel_val, cnn_val_pred)
print(f"\n>>> CNN-BiLSTM MAE on Validation Set: {cnn_mae:.4f}")

## Model Training, Hyperparameter Tuning, and Final Comparison

This cell constructs the machine learning pipelines and systematically evaluates a suite of regression algorithms across the different acoustic representations (Handcrafted, Wav2Vec 2.0, and AST). It concludes by comparing the best-performing traditional pipelines against the deep learning CNN-BiLSTM baseline.

**Main operations performed in this block:**

1. **Robust Pipeline Construction:** Defines leak-free Scikit-Learn pipelines that integrate median imputation and standard scaling. This ensures that preprocessing transformations are fitted exclusively within the cross-validation folds, preventing data leakage.
2. **Algorithm and Hyperparameter Grid:** Establishes a diverse model search space comprising linear models (Ridge), instance-based learning (KNN), support vector machines (SVR with RBF kernel), and tree-based ensembles (Decision Trees, Random Forests, Gradient Boosting).
3. **Training and Validation (Grid Search):** Implements an evaluation protocol that uses Grid Search Cross-Validation (GridSearchCV) on the training split to discover optimal hyperparameters for each algorithm and representation type, subsequently scoring the best models on the held-out validation set.
4. **Comprehensive Evaluation:** Computes standard regression and correlation metrics (MSE, MAE, Pearson, and Spearman coefficients) to benchmark the predictive power of handcrafted features versus deep SSL embeddings.
5. **Final Unified Comparison:** Identifies the top-performing model for each feature representation and aggregates them into a final summary table, directly comparing their performance against the CNN-BiLSTM validation metrics computed in the previous step.

In [ ]:
# MODELING (Classic, Wav2Vec2, AST)
def get_metrics(y_true, y_pred):
    """Computes MSE, MAE, Pearson and Spearman correlations."""
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    if len(y_true) > 1 and np.std(y_pred) > 1e-12 and np.std(y_true) > 1e-12:
        pearson = float(pearsonr(y_true, y_pred)[0])
        spearman = float(spearmanr(y_true, y_pred)[0])
    else:
        pearson, spearman = 0.0, 0.0
    return {"MSE": float(mse), "MAE": float(mae), "Pearson": pearson, "Spearman": spearman}

def add_awgn(waveform, snr_db):
    """Adds Additive White Gaussian Noise at the requested SNR (dB)."""
    waveform = np.asarray(waveform, dtype=np.float32)
    signal_power = np.mean(waveform ** 2) + 1e-12
    snr_linear = 10.0 ** (snr_db / 10.0)
    noise_power = signal_power / snr_linear
    noise = np.random.normal(0.0, np.sqrt(noise_power), size=waveform.shape).astype(np.float32)
    return (waveform + noise).astype(np.float32)

def build_pipeline(estimator):
    """Leak-free pipeline: imputation and scaling are fitted inside cross validation."""
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", estimator)
    ])

def build_model_space():
    return {
        "Ridge": (Ridge(random_state=SEED),
                  {"model__alpha": [1.0, 10.0]}),
        "KNN": (KNeighborsRegressor(),
                {"model__n_neighbors": [5, 11]}),
        "DecisionTree": (DecisionTreeRegressor(random_state=SEED),
                         {"model__max_depth": [10, None]}),
        "RandomForest": (RandomForestRegressor(random_state=SEED),
                         {"model__n_estimators": [100], "model__max_depth": [10]}),
        "GradientBoosting": (GradientBoostingRegressor(random_state=SEED),
                             {"model__n_estimators": [100], "model__learning_rate": [0.1], "model__max_depth": [3]}),
        "SVR-RBF": (SVR(kernel="rbf"),
                    {"model__C": [1.0, 10.0], "model__gamma": ["scale"]}),
    }

def nested_cv_evaluate(X, y, representation, groups=None, n_outer=5, n_inner=3):
    """Nested Cross Validation: inner GridSearchCV tuning, outer unbiased evaluation."""
    X = np.asarray(X)
    y = np.asarray(y, dtype=float)
    if groups is not None:
        outer_splits = list(GroupKFold(n_splits=n_outer).split(X, y, groups))
    else:
        outer_splits = list(KFold(n_splits=n_outer, shuffle=True, random_state=SEED).split(X, y))

    rows = []
    for name, (estimator, grid) in build_model_space().items():
        fold_metrics = []
        for tr_idx, te_idx in outer_splits:
            inner = KFold(n_splits=n_inner, shuffle=True, random_state=SEED)
            search = GridSearchCV(build_pipeline(estimator), grid, scoring="neg_mean_absolute_error", cv=inner, n_jobs=-1)
            search.fit(X[tr_idx], y[tr_idx])
            pred = search.predict(X[te_idx])
            fold_metrics.append(get_metrics(y[te_idx], pred))
        row = {"Representation": representation, "Model": name}
        for metric in ["MSE", "MAE", "Pearson", "Spearman"]:
            row[metric] = float(np.mean([m[metric] for m in fold_metrics]))
            row[metric + "_std"] = float(np.std([m[metric] for m in fold_metrics]))
        rows.append(row)
    return pd.DataFrame(rows)

def fit_and_evaluate(X_tr, y_tr, X_va, y_va, representation, n_inner=5):
    """Tunes each model on train (inner CV) and evaluates on the held-out validation set."""
    trained = {}
    rows = []

    models = build_model_space()
    for name, (estimator, grid) in tqdm(models.items(), desc=f"Training models ({representation})"):
        inner = KFold(n_splits=n_inner, shuffle=True, random_state=SEED)
        search = GridSearchCV(build_pipeline(estimator), grid, scoring="neg_mean_absolute_error", cv=inner, n_jobs=-1)
        search.fit(X_tr, y_tr)
        best = search.best_estimator_
        trained[name] = best
        pred = best.predict(X_va)
        row = {"Representation": representation, "Model": name}
        row.update(get_metrics(y_va, pred))
        rows.append(row)
    return trained, pd.DataFrame(rows)

# Representation matrices aligned to filtered_model_df (1000 samples)
splits = filtered_model_df["split"].to_numpy()
y_all = filtered_model_df["target_score"].to_numpy().astype(float)
train_mask = splits == "train"
val_mask = splits == "validation"
test_mask = splits == "test"
dev_mask = train_mask | val_mask

X_classic_df = filtered_model_df[feature_columns].copy()
X_train = X_classic_df.loc[train_mask].reset_index(drop=True)
X_val = X_classic_df.loc[val_mask].reset_index(drop=True)
X_test = X_classic_df.loc[test_mask].reset_index(drop=True)
y_train = y_all[train_mask]
y_val = y_all[val_mask]
y_test = y_all[test_mask]

Xw_train, Xw_val, Xw_test = w2v_all[train_mask], w2v_all[val_mask], w2v_all[test_mask]
Xa_train, Xa_val, Xa_test = ast_all[train_mask], ast_all[val_mask], ast_all[test_mask]

# Final tuned models fitted on train, scored on validation
print("\nTRAIN / VALIDATION MODEL COMPARISON")
trained_models, classic_metrics = fit_and_evaluate(X_train, y_train, X_val, y_val, "Classic")
trained_models_w2v, w2v_metrics = fit_and_evaluate(Xw_train, y_train, Xw_val, y_val, "Wav2Vec2")
trained_models_ast, ast_metrics = fit_and_evaluate(Xa_train, y_train, Xa_val, y_val, "AST")

representation_comparison = pd.concat([classic_metrics, w2v_metrics, ast_metrics], ignore_index=True)
representation_comparison.to_csv(RESULTS_DIR / "representation_comparison.csv", index=False)
display(representation_comparison.round(4))

def best_row(metrics_df):
    return metrics_df.sort_values("MAE").iloc[0]

best_classic_row = best_row(classic_metrics)
best_w2v_row = best_row(w2v_metrics)
best_ast_row = best_row(ast_metrics)

best_classic_name = best_classic_row["Model"]
best_w2v_name = best_w2v_row["Model"]
best_ast_name = best_ast_row["Model"]

best_classic_model = trained_models[best_classic_name]
best_w2v_model = trained_models_w2v[best_w2v_name]
best_ast_model = trained_models_ast[best_ast_name]

# Final table: Classic vs Wav2Vec2 vs AST vs CNN-BiLSTM (baseline)
cnn_metrics = get_metrics(y_mel_val, cnn_val_pred)

final_comparison_df = pd.DataFrame([
    {"Representation": "Classic Features", "Best Model": best_classic_name, **{k: best_classic_row[k] for k in ["MSE", "MAE", "Pearson", "Spearman"]}},
    {"Representation": "Wav2Vec2", "Best Model": best_w2v_name, **{k: best_w2v_row[k] for k in ["MSE", "MAE", "Pearson", "Spearman"]}},
    {"Representation": "AST", "Best Model": best_ast_name, **{k: best_ast_row[k] for k in ["MSE", "MAE", "Pearson", "Spearman"]}},
    {"Representation": "CNN-BiLSTM (baseline)", "Best Model": "CNN-BiLSTM", **cnn_metrics},
])
final_comparison_df.to_csv(RESULTS_DIR / "final_representation_comparison.csv", index=False)
print("\nFINAL REPRESENTATION COMPARISON")
display(final_comparison_df.round(4))

In [ ]:
# Retrieve the target scores for the Test set
y_mel_test = cnn_df.loc[cnn_df["split"] == "test", "target_score"].to_numpy()

# Normalize Test Set spectrograms using Train mean and variance
X_mel_test = ((X_mel_test_raw - mel_train_mean) / mel_train_std)[..., np.newaxis]

# Generate CNN predictions on the Test Set
cnn_test_pred = cnn_bilstm.predict(X_mel_test, batch_size=32, verbose=0).ravel()

# Calculate CNN metrics
cnn_test_metrics = get_metrics(y_mel_test, cnn_test_pred)

# CREATE FINAL TABLE (Test Set)
final_test_comparison_df = pd.DataFrame([
    {"Representation": "Classic Features", "Best Model": best_classic_name, **get_metrics(y_test, best_classic_model.predict(X_test))},
    {"Representation": "Wav2Vec2", "Best Model": best_w2v_name, **get_metrics(y_test, best_w2v_model.predict(Xw_test))},
    {"Representation": "AST", "Best Model": best_ast_name, **get_metrics(y_test, best_ast_model.predict(Xa_test))},
    {"Representation": "CNN-BiLSTM (baseline)", "Best Model": "CNN-BiLSTM", **cnn_test_metrics},
])

print("\nFINAL TEST SET COMPARISON")
display(final_test_comparison_df.round(4))

# GENERATE 2x2 DASHBOARD
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = [
    {"name": "MSE", "ax": axes[0, 0], "title": "Test MSE - Lower is Better", "palette": "mako"},
    {"name": "MAE", "ax": axes[0, 1], "title": "Test MAE - Lower is Better", "palette": "mako"},
    {"name": "Pearson", "ax": axes[1, 0], "title": "Test Pearson Correlation - Higher is Better", "palette": "flare"},
    {"name": "Spearman", "ax": axes[1, 1], "title": "Test Spearman Correlation - Higher is Better", "palette": "flare"}
]

for m in metrics:
    sns.barplot(
        data=final_test_comparison_df,
        x="Representation",
        y=m["name"],
        ax=m["ax"],
        hue="Representation",
        palette=m["palette"],
        legend=False
    )
    m["ax"].set_title(m["title"], fontsize=14, fontweight="bold")
    m["ax"].set_ylabel(m["name"])
    m["ax"].set_xlabel("")
    m["ax"].tick_params(axis='x', rotation=15)

    if m["name"] in ["Pearson", "Spearman"]:
        m["ax"].set_ylim(0, 1.0)

    for p in m["ax"].patches:
        m["ax"].annotate(f"{p.get_height():.4f}",
                         (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='center',
                         xytext=(0, 8), textcoords='offset points',
                         fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "final_test_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

## Model Stability Evaluation and Explainable AI (SHAP)

This cell assesses the robustness of the best-performing traditional model through multi-seed testing and interprets its decision-making process using SHAP (SHapley Additive exPlanations) to identify which acoustic features drive the attractiveness predictions.

**Main operations performed in this block:**

1. **Multi-Seed Stability Testing:** Tests the consistency of the chosen model (`SVR-RBF`) by cloning its pipeline and retraining it across multiple random seeds. It calculates the mean and standard deviation of the evaluation metrics to measure variance and ensure the results are reliable.
2. **SHAP Background Preparation:** Subsamples both the training and validation datasets to create a computationally manageable background distribution, which is required to calculate the baseline predictions.
3. **Permutation Explainer:** Initializes a SHAP explainer utilizing the permutation algorithm. This approach is specifically tailored for interpreting non-tree, black-box models like Support Vector Machines.
4. **Feature Importance Visualization:** Computes the mean absolute SHAP values to quantify the overall impact of each handcrafted acoustic feature. It then generates and saves a bar chart displaying the top 20 most influential variables, providing transparency into what the model prioritizes when scoring human voice attractiveness.

In [ ]:
# Statistical Multi-Seed Evaluation
N_SEEDS = [7, 19, 42, 73, 101]
repeat_results = []

# Select the best classic model
best_model_name = "SVR-RBF"
best_pipeline = trained_models[best_model_name]

print(f"Multi-seed stability evaluation for {best_model_name}")
for run_seed in tqdm(N_SEEDS, desc="Multi-Seed Experiments"):
    # Clone the model architecture to start from scratch
    repeated_model = clone(best_pipeline)

    # If the model has a random_state parameter, we set it
    if "model__random_state" in repeated_model.get_params():
        repeated_model.set_params(model__random_state=run_seed)

    repeated_model.fit(X_train, y_train)
    prediction = repeated_model.predict(X_val)

    metrics = {"seed": run_seed, "Model": best_model_name}
    metrics.update(get_metrics(y_val, prediction))
    repeat_results.append(metrics)

repeat_df = pd.DataFrame(repeat_results)
repeat_summary = repeat_df.groupby("Model").agg(
    MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
    Pearson_mean=("Pearson", "mean"), Pearson_std=("Pearson", "std")
).round(4)

display(repeat_summary)

# XAI with SHAP
print(f"\nSHAP Feature Importance Extraction for {best_model_name}")

# We use a subsample for computational reasons
X_shap = X_val.sample(min(120, len(X_val)), random_state=SEED).copy()
X_shap_bg = shap.sample(X_train, min(80, len(X_train)), random_state=SEED)

# We use the permutation algorithm for non-tree black-box models
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    shap_explainer = shap.Explainer(
        best_pipeline.predict,
        X_shap_bg,
        algorithm="permutation"
    )

    # Calculate SHAP values
    shap_output = shap_explainer(X_shap, max_evals=2 * len(feature_columns) + 1)

shap_values = shap_output.values
if isinstance(shap_values, list):
    shap_values = shap_values[0]

# Create a summary DataFrame
shap_imp_df = pd.DataFrame({
    "feature": feature_columns,
    "mean_absolute_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_absolute_shap", ascending=False).reset_index(drop=True)

# Plot Top 20 Features
plt.figure(figsize=(10, 8))
sns.barplot(
    data=shap_imp_df.head(20).sort_values("mean_absolute_shap"),
    x="mean_absolute_shap",
    y="feature",
    hue="feature", # Added to fix Seaborn warning
    palette="magma",
    legend=False   # Added to keep the plot clean
)
plt.title(f"SHAP: Top 20 most important features for {best_model_name}")
plt.xlabel("Mean absolute impact on attractiveness prediction")
plt.ylabel("")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "shap_SVR-RBF.png", dpi=150, bbox_inches="tight")
plt.show()

## Noise Robustness Stress Testing

This cell evaluates how well each feature representation (Handcrafted, Wav2Vec 2.0, and AST) handles acoustic degradation by simulating noisy real-world environments. It tests the resilience of the previously trained best models against varying levels of background noise.

**Main operations performed in this block:**

1. **Noise Simulation (AWGN):** Defines multiple Signal-to-Noise Ratio (SNR) levels, ranging from a clean baseline (infinite SNR) down to severe acoustic degradation (5 dB), and progressively injects Additive White Gaussian Noise into a subset of the test audio tracks.
2. **Dynamic Feature Re-extraction:** Utilizes a secure temporary directory to save the noisy audio files and independently re-extracts both the traditional acoustic features and the deep embeddings (Wav2Vec2 and AST) directly from these degraded signals.
3. **Inference Under Stress:** Passes the newly extracted, noisy feature vectors through the corresponding best pre-trained pipelines to generate attractiveness predictions. It then calculates the error metrics (MAE and MSE) against the ground truth scores.
4. **Degradation Visualization:** Aggregates the performance metrics and generates comparative line plots. These charts visually demonstrate the performance decay of each representation as noise severity increases, highlighting which feature extraction method is most robust to acoustic interference.

In [ ]:
# Noise robustness stress test (Classic vs Wav2Vec2 vs AST)
print("\nRobustness stress test (AWGN + Feature/Embedding Re-extraction)")

# Define noise levels to observe degradation
snr_levels = [np.inf, 30, 20, 10, 5]
robust_metrics = []

test_subset = filtered_model_df[filtered_model_df["split"] == "test"].sample(min(60, len(filtered_model_df[filtered_model_df["split"] == "test"])), random_state=SEED)

# Use a temporary directory to avoid cluttering the disk
with tempfile.TemporaryDirectory() as temp_dir:
    temp_dir_path = Path(temp_dir)

    for snr in tqdm(snr_levels, desc="SNR Test (Increasing severity)"):
        y_true = []
        pred_classic, pred_w2v, pred_ast = [], [], []

        for idx, row in test_subset.iterrows():
            # Load original audio
            waveform, sr = librosa.load(row["audio_path"], sr=SAMPLE_RATE, mono=True)

            # Add noise if not "clean" (inf)
            if not np.isinf(snr):
                waveform = add_awgn(waveform, snr)

            # Save temporarily and RE-EXTRACT both classic features and deep embeddings
            tmp_path = temp_dir_path / f"temp_{idx}.wav"
            sf.write(tmp_path, waveform, SAMPLE_RATE)

            noisy_features = extract_handcrafted_features(str(tmp_path))
            features_df = pd.DataFrame([noisy_features]).reindex(columns=feature_columns)
            w2v_emb, ast_emb = extract_deep_embeddings(str(tmp_path))

            # Prediction with the best model of each representation
            y_true.append(row["target_score"])
            pred_classic.append(float(best_classic_model.predict(features_df)[0]))
            pred_w2v.append(float(best_w2v_model.predict(w2v_emb.reshape(1, -1))[0]))
            pred_ast.append(float(best_ast_model.predict(ast_emb.reshape(1, -1))[0]))

        snr_label = "Clean" if np.isinf(snr) else str(snr)
        for rep_name, preds in [("Classic", pred_classic), ("Wav2Vec2", pred_w2v), ("AST", pred_ast)]:
            robust_metrics.append({
                "SNR_dB": snr_label,
                "Representation": rep_name,
                "MAE": mean_absolute_error(y_true, preds),
                "MSE": mean_squared_error(y_true, preds)
            })

# Result visualization
robustness_df = pd.DataFrame(robust_metrics)
snr_order = ["Clean", "30", "20", "10", "5"]
robustness_df["SNR_dB"] = pd.Categorical(robustness_df["SNR_dB"], categories=snr_order, ordered=True)
robustness_df = robustness_df.sort_values(["Representation", "SNR_dB"]).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.lineplot(data=robustness_df, x="SNR_dB", y="MAE", hue="Representation", marker="o", linewidth=2, ax=axes[0])
axes[0].set_title("MAE degradation under AWGN noise")
axes[0].set_xlabel("Noise Level (SNR in dB, Clean = No noise)")
axes[0].set_ylabel("MAE (higher = worse)")
axes[0].grid(True, linestyle="--", alpha=0.6)

sns.lineplot(data=robustness_df, x="SNR_dB", y="MSE", hue="Representation", marker="o", linewidth=2, ax=axes[1])
axes[1].set_title("MSE degradation under AWGN noise")
axes[1].set_xlabel("Noise Level (SNR in dB, Clean = No noise)")
axes[1].set_ylabel("MSE (higher = worse)")
axes[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "robustness_comparison.png", dpi=150, bbox_inches="tight")
plt.show()